# CNN From Scratch NumPy

Nguyễn Ngọc Hoàng Nam - B23DCCN585 | Assignment 04

**Bản mở rộng:** notebook này giữ nhóm 18 lượt seed 42 để giải thích thuật toán. Notebook **06** tổng hợp đầy đủ 54 lượt nhiều seed và 18 ablation; notebook **07** thực thi ví dụ số học dùng trong báo cáo 78 trang.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Hãy mở notebook từ thư mục repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.data import DATASETS, load_data, data_root
print('Python:', sys.version.split()[0])
print('Dữ liệu:', data_root())

Python: 3.10.20
Dữ liệu: E:\PTHTTM\ASG_04_data


## 1. Cách triển khai

Toàn bộ convolution, pooling, dense, BatchNorm, residual, dropout, cross-entropy và Adam được viết bằng NumPy. Không dùng autograd hoặc lớp mạng của framework trong mô hình này.

Mã trong các ô dưới lấy trực tiếp từ module trong `src/`. Vòng lặp huấn luyện lưu checkpoint theo validation loss và chỉ đánh giá test sau khi khôi phục checkpoint tốt nhất.

In [2]:
BACKEND = 'numpy'
RETRAIN = False  # Đổi thành True để huấn luyện lại 6 cấu hình; sẽ ghi đè kết quả tương ứng.

## 2. Mã mô hình

In [3]:
"""CNN from scratch. All forward/backward operations and Adam updates use NumPy.

Layout is NCHW. Convolution implements cross-correlation (as in both frameworks).
No autograd, torch or TensorFlow is used in this module.
"""
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


class Layer:
    def params(self): return []
    def state(self): return {}


class Conv2D(Layer):
    def __init__(self, cin, cout, rng, kernel=3, padding=1):
        self.k, self.p = kernel, padding
        self.w = (rng.standard_normal((cout, cin, kernel, kernel)) * np.sqrt(2/(cin*kernel*kernel))).astype('float32')
        self.b = np.zeros(cout, dtype='float32')
    def forward(self, x, training=True):
        n,c,h,w = x.shape
        xp = np.pad(x, ((0,0),(0,0),(self.p,self.p),(self.p,self.p)))
        win = sliding_window_view(xp, (self.k,self.k), axis=(2,3))
        oh,ow = win.shape[2:4]
        cols = win.transpose(0,2,3,1,4,5).reshape(n*oh*ow,-1)
        if training: self.cache = (x.shape, cols, oh, ow)
        y = cols @ self.w.reshape(len(self.w),-1).T + self.b
        return y.reshape(n,oh,ow,-1).transpose(0,3,1,2)
    def backward(self, dy):
        shape,cols,oh,ow = self.cache
        n,c,h,w = shape
        g = dy.transpose(0,2,3,1).reshape(-1,len(self.w))
        self.dw = (g.T @ cols).reshape(self.w.shape)
        self.db = g.sum(axis=0)
        dc = (g @ self.w.reshape(len(self.w),-1)).reshape(n,oh,ow,c,self.k,self.k)
        dxp = np.zeros((n,c,h+2*self.p,w+2*self.p),dtype='float32')
        for u in range(self.k):
            for v in range(self.k):
                dxp[:,:,u:u+oh,v:v+ow] += dc[:,:,:,:,u,v].transpose(0,3,1,2)
        return dxp[:,:,self.p:self.p+h,self.p:self.p+w]
    def params(self): return [(self.w,self.dw),(self.b,self.db)]
    def state(self): return {'weight':self.w, 'bias':self.b}


class BatchNorm2D(Layer):
    def __init__(self, channels, eps=1e-5, momentum=0.1):
        self.gamma=np.ones(channels,dtype='float32');self.beta=np.zeros(channels,dtype='float32')
        self.mean=np.zeros(channels,dtype='float32');self.var=np.ones(channels,dtype='float32')
        self.eps,self.momentum=eps,momentum
    def forward(self,x,training=True):
        axes=(0,2,3)
        if training:
            mean=x.mean(axis=axes);var=x.var(axis=axes)
            # Population variance is used in all three implementations.
            self.mean *= 1-self.momentum;self.mean += self.momentum*mean
            self.var *= 1-self.momentum;self.var += self.momentum*var
        else: mean,var=self.mean,self.var
        inv=1/np.sqrt(var+self.eps)
        z=(x-mean[None,:,None,None])*inv[None,:,None,None]
        if training:self.cache=(z,inv)
        return z*self.gamma[None,:,None,None]+self.beta[None,:,None,None]
    def backward(self,g):
        z,inv=self.cache;axes=(0,2,3)
        self.dg=(g*z).sum(axis=axes);self.db=g.sum(axis=axes)
        dx=(g-g.mean(axis=axes,keepdims=True)-z*(g*z).mean(axis=axes,keepdims=True))
        return dx*(self.gamma*inv)[None,:,None,None]
    def params(self):return [(self.gamma,self.dg),(self.beta,self.db)]
    def state(self):return {'gamma':self.gamma,'beta':self.beta,'running_mean':self.mean,'running_var':self.var}


class ReLU(Layer):
    def forward(self,x,training=True):
        if training:self.mask=x>0
        return np.maximum(x,0)
    def backward(self,g):return g*self.mask


class BatchNorm1D(BatchNorm2D):
    """The same manual normalization over the batch for Dense features."""
    def forward(self,x,training=True):
        return super().forward(x[:,:,None,None],training)[:,:,0,0]
    def backward(self,g):
        return super().backward(g[:,:,None,None])[:,:,0,0]


class MaxPool2D(Layer):
    def forward(self,x,training=True):
        n,c,h,w=x.shape;oh,ow=h//2,w//2
        windows=x[:,:,:oh*2,:ow*2].reshape(n,c,oh,2,ow,2).transpose(0,1,2,4,3,5).reshape(n,c,oh,ow,4)
        if training:self.cache=(x.shape,windows.argmax(axis=-1))
        return windows.max(axis=-1)
    def backward(self,g):
        shape,idx=self.cache;n,c,h,w=shape;oh,ow=h//2,w//2
        out=np.zeros((*g.shape,4),dtype='float32')
        np.put_along_axis(out,idx[...,None],g[...,None],axis=-1)
        out=out.reshape(n,c,oh,ow,2,2).transpose(0,1,2,4,3,5).reshape(n,c,oh*2,ow*2)
        dx=np.zeros(shape,dtype='float32');dx[:,:,:oh*2,:ow*2]=out
        return dx


class Flatten(Layer):
    def forward(self,x,training=True):
        if training:self.shape=x.shape
        return x.reshape(len(x),-1)
    def backward(self,g):return g.reshape(self.shape)


class Dense(Layer):
    def __init__(self,cin,cout,rng):
        self.w=(rng.standard_normal((cin,cout))*np.sqrt(2/cin)).astype('float32')
        self.b=np.zeros(cout,dtype='float32')
    def forward(self,x,training=True):
        if training:self.x=x
        return x@self.w+self.b
    def backward(self,g):
        self.dw=self.x.T@g;self.db=g.sum(axis=0)
        return g@self.w.T
    def params(self):return [(self.w,self.dw),(self.b,self.db)]
    def state(self):return {'weight':self.w,'bias':self.b}


class Dropout(Layer):
    def __init__(self,rng,rate=0.25):self.rng,self.rate=rng,rate
    def forward(self,x,training=True):
        if not training:return x
        self.mask=(self.rng.random(x.shape)>=self.rate).astype('float32')/(1-self.rate)
        return x*self.mask
    def backward(self,g):return g*self.mask


class Residual(Layer):
    """ReLU(BN(Conv(x)) + x), preserving the shape and adding both gradients."""
    def __init__(self,channels,rng):
        self.conv=Conv2D(channels,channels,rng);self.bn=BatchNorm2D(channels);self.relu=ReLU()
    def forward(self,x,training=True):
        return self.relu.forward(self.bn.forward(self.conv.forward(x,training),training)+x,training)
    def backward(self,g):
        g=self.relu.backward(g)
        return g+self.conv.backward(self.bn.backward(g))
    def params(self):return self.conv.params()+self.bn.params()
    def state(self):return {**{'conv_'+k:v for k,v in self.conv.state().items()},**{'bn_'+k:v for k,v in self.bn.state().items()}}


class CNN:
    def __init__(self,channels,size,classes,variant='baseline',seed=42):
        rng=np.random.default_rng(seed);improved=variant=='improved'
        self.layers=[Conv2D(channels,8,rng)]
        if improved:self.layers.append(BatchNorm2D(8))
        self.layers += [ReLU(),MaxPool2D(),Conv2D(8,16,rng)]
        if improved:self.layers.append(BatchNorm2D(16))
        self.layers.append(ReLU())
        if improved:self.layers.append(Residual(16,rng))
        self.layers += [MaxPool2D(),Flatten(),Dense(16*(size//4)**2,64,rng)]
        if improved:self.layers.append(BatchNorm1D(64))
        self.layers.append(ReLU())
        if improved:self.layers.append(Dropout(rng))
        self.layers.append(Dense(64,classes,rng))
    def forward(self,x,training=True):
        for layer in self.layers:x=layer.forward(x,training)
        return x
    def backward(self,g):
        for layer in reversed(self.layers):g=layer.backward(g)
        return g
    def params(self):return [p for layer in self.layers for p in layer.params()]
    def state(self):return {f'{i}.{k}':v.copy() for i,layer in enumerate(self.layers) for k,v in layer.state().items()}
    def load_state(self,state):
        for i,layer in enumerate(self.layers):
            for k,v in layer.state().items():v[:]=state[f'{i}.{k}']
    def parameter_count(self):
        return sum(v.size for layer in self.layers for k,v in layer.state().items() if not k.endswith(('running_mean','running_var')))


def cross_entropy(logits,labels):
    z=logits-logits.max(axis=1,keepdims=True)
    logsum=np.log(np.exp(z).sum(axis=1,keepdims=True))
    loss=float((logsum[:,0]-z[np.arange(len(labels)),labels]).mean())
    grad=np.exp(z-logsum)
    grad[np.arange(len(labels)),labels]-=1
    return loss,grad/len(labels)


class Adam:
    def __init__(self,lr=1e-3,beta1=0.9,beta2=0.999,eps=1e-8):
        self.lr,self.b1,self.b2,self.eps=lr,beta1,beta2,eps;self.t=0;self.m=[];self.v=[]
    def step(self,params):
        if not self.m:
            self.m=[np.zeros_like(w) for w,g in params];self.v=[np.zeros_like(w) for w,g in params]
        self.t+=1
        for (w,g),m,v in zip(params,self.m,self.v):
            m*=self.b1;m+=(1-self.b1)*g
            v*=self.b2;v+=(1-self.b2)*g*g
            w-=self.lr*(m/(1-self.b1**self.t))/(np.sqrt(v/(1-self.b2**self.t))+self.eps)


## 3. Vòng lặp huấn luyện và đánh giá

Chương trình bên dưới chứa đầy đủ bước lấy batch, forward, loss, gradient, cập nhật, validation, checkpoint và đánh giá test. Các lượt chạy thực tế gọi cùng module trong một tiến trình riêng để cô lập framework.

In [4]:
"""Reproducible full-dataset training. Run: python -m src.train --help."""
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(key,'1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
import json,csv,time,platform,argparse,sys,copy
from pathlib import Path
import numpy as np
from src.data import ROOT,DATASETS,load_data,batches
from src.numpy_cnn import CNN,cross_entropy,Adam

def classification_metrics(y,logits,k):
    pred=logits.argmax(axis=1);cm=np.bincount(y*k+pred,minlength=k*k).reshape(k,k)
    tp=np.diag(cm).astype(float)
    precision=np.divide(tp,cm.sum(0),out=np.zeros(k),where=cm.sum(0)>0)
    recall=np.divide(tp,cm.sum(1),out=np.zeros(k),where=cm.sum(1)>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(k),where=(precision+recall)>0)
    result={'accuracy':float((pred==y).mean()),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'test_loss':cross_entropy(logits,y)[0]}
    if k>=5:result['top5_accuracy']=float(np.any(np.argsort(logits,axis=1)[:,-5:]==y[:,None],axis=1).mean())
    return result,cm

def experiment_dir(backend,dataset,variant,seed=42,ablation=None):
    name=f'{dataset}_{backend}_{variant}'
    if ablation:return ROOT/'results'/'ablation'/f'seed_{seed}'/(name+'_'+ablation)
    if seed!=42:return ROOT/'results'/'multiseed'/f'seed_{seed}'/name
    return ROOT/'results'/name

def _train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    if ablation and (backend!='pytorch' or variant!='improved'):
        raise ValueError('Ablation requires the PyTorch improved architecture.')
    cfg=DATASETS[dataset];epochs=epochs or cfg['epochs']
    out=experiment_dir(backend,dataset,variant,seed,ablation);out.mkdir(parents=True,exist_ok=True)
    if (out/'metrics.json').exists() and not force:
        saved=json.loads((out/'config.json').read_text())
        for key,value in dict(seed=seed,epochs=epochs,batch_size=batch_size,ablation=ablation).items():
            if saved.get(key)!=value:raise ValueError(f'Existing {out}: {key} differs; use --force or another seed.')
        return json.loads((out/'metrics.json').read_text())
    data=load_data(dataset);x,y=data['x'],data['y'];ti,vi=data['train_ids'],data['val_ids']
    spec=dict(channels=cfg['channels'],size=cfg['size'],classes=cfg['classes'],variant=variant,seed=seed)
    initial=CNN(**spec);state=initial.state();params=initial.parameter_count()
    np.random.seed(seed)
    if backend=='numpy':
        model=initial;optimizer=Adam();device='CPU';version=np.__version__
        def predict(b):return model.forward(b,False)
        def step(b,t):
            logits=model.forward(b,True);loss,g=cross_entropy(logits,t);model.backward(g);optimizer.step(model.params())
            return loss,int((logits.argmax(1)==t).sum())
        def save():np.savez_compressed(out/'weights.npz',**model.state())
        def restore():
            with np.load(out/'weights.npz') as d:model.load_state(dict(d))
    elif backend=='pytorch':
        import torch
        from src.torch_cnn import TorchCNN
        torch.manual_seed(seed);torch.set_num_threads(4)
        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
        device='cuda' if torch.cuda.is_available() else 'cpu';version=torch.__version__
        model=TorchCNN(cfg['channels'],cfg['size'],cfg['classes'],variant);model.load_numpy(state);model.to(device)
        assert sum(p.numel() for p in model.parameters())==params
        model.ablate(ablation);params=sum(p.numel() for p in model.parameters())
        optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,eps=1e-8)
        def predict(b):
            model.eval()
            with torch.no_grad():return model(torch.from_numpy(np.ascontiguousarray(b)).to(device)).cpu().numpy()
        def step(b,t):
            model.train();optimizer.zero_grad(set_to_none=True)
            logits=model(torch.from_numpy(np.ascontiguousarray(b)).to(device));target=torch.from_numpy(t).to(device)
            loss=torch.nn.functional.cross_entropy(logits,target);loss.backward();optimizer.step()
            return loss.item(),int((logits.argmax(1)==target).sum().item())
        def save():torch.save(model.state_dict(),out/'weights.pt')
        def restore():model.load_state_dict(torch.load(out/'weights.pt',map_location=device,weights_only=True))
    elif backend=='tensorflow':
        cuda=os.environ.get('CNN_CUDA_DIR','E:/PTHTTM/ASG_04_runtime/cuda/Library/bin')
        if os.name=='nt' and Path(cuda).is_dir():
            os.environ['PATH']=cuda+os.pathsep+os.environ['PATH'];dll=os.add_dll_directory(cuda)
        import tensorflow as tf
        from src.tf_cnn import build_model,load_numpy
        tf.keras.utils.set_random_seed(seed)
        tf.config.threading.set_intra_op_parallelism_threads(4);tf.config.threading.set_inter_op_parallelism_threads(2)
        tf.config.experimental.enable_tensor_float_32_execution(False)
        gpus=tf.config.list_physical_devices('GPU')
        for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
        device='GPU' if gpus else 'CPU';version=tf.__version__
        model=build_model(cfg['channels'],cfg['size'],cfg['classes'],variant);load_numpy(model,state,variant)
        assert sum(int(np.prod(v.shape)) for v in model.trainable_weights)==params
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3,epsilon=1e-8)
        @tf.function(reduce_retracing=True)
        def train_step(b,t):
            with tf.GradientTape() as tape:
                logits=model(b,training=True)
                loss=tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=t,logits=logits))
            optimizer.apply_gradients(zip(tape.gradient(loss,model.trainable_weights),model.trainable_weights))
            return loss,tf.reduce_sum(tf.cast(tf.argmax(logits,axis=1)==t,tf.int32))
        @tf.function(reduce_retracing=True)
        def infer(b):return model(b,training=False)
        def predict(b):return infer(np.ascontiguousarray(b.transpose(0,2,3,1))).numpy()
        def step(b,t):
            loss,correct=train_step(np.ascontiguousarray(b.transpose(0,2,3,1)),t)
            return float(loss.numpy()),int(correct.numpy())
        def save():model.save_weights(str(out/'weights.h5'))
        def restore():model.load_weights(str(out/'weights.h5'))
    else:raise ValueError(backend)
    history=[];best=float('inf');best_epoch=0;train_seconds=0;val_seconds=0
    print(f'START {dataset} {backend} {variant} seed={seed} ablation={ablation}: train={len(ti)} val={len(vi)} epochs={epochs} device={device} params={params}',flush=True)
    config=dict(dataset=dataset,backend=backend,variant=variant,epochs=epochs,batch_size=batch_size,seed=seed,learning_rate=0.001,optimizer='Adam',adam_beta1=0.9,adam_beta2=0.999,adam_epsilon=1e-8,normalization='uint8 / 255',train_samples=len(ti),validation_samples=len(vi),test_samples=len(data['y_test']),device=device,framework_version=version,python=sys.version,platform=platform.platform(),parameter_count=params,selection='minimum validation cross-entropy',augmentation=False,training_scope='all predefined training samples, no subsampling')
    from threadpoolctl import threadpool_info
    config['ablation']=ablation
    config['split_seed']=42
    config['cpu_thread_pools']=threadpool_info()
    (out/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    for epoch in range(1,epochs+1):
        ts=time.perf_counter();ls=0;correct=0
        for b,t in batches(x,y,ti,batch_size,seed+epoch):
            loss,c=step(b,t);ls+=loss*len(t);correct+=c
        elapsed=time.perf_counter()-ts;train_seconds+=elapsed;ts=time.perf_counter()
        logits=np.concatenate([predict(b) for b,t in batches(x,y,vi,batch_size)])
        vl=cross_entropy(logits,y[vi])[0];va=float((logits.argmax(1)==y[vi]).mean());ve=time.perf_counter()-ts;val_seconds+=ve
        if not np.isfinite(ls+vl):raise FloatingPointError('Non-finite loss. Stop rather than record invalid results.')
        if vl<best:best=vl;best_epoch=epoch;save()
        row=dict(epoch=epoch,train_loss=ls/len(ti),train_accuracy=correct/len(ti),val_loss=vl,val_accuracy=va,train_seconds=elapsed,validation_seconds=ve)
        history.append(row)
        with (out/'history.csv').open('w',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=list(row));writer.writeheader();writer.writerows(history)
        print(f'{dataset}/{backend}/{variant} epoch {epoch}/{epochs}: loss {row["train_loss"]:.4f} val_loss {vl:.4f} val_acc {va:.4f} train_s {elapsed:.1f}',flush=True)
    restore();xt,yt=data['x_test'],data['y_test'];ts=time.perf_counter()
    logits=np.concatenate([predict(b) for b,t in batches(xt,yt,np.arange(len(yt)),batch_size)])
    infer_seconds=time.perf_counter()-ts
    metrics,cm=classification_metrics(yt,logits,cfg['classes'])
    metrics.update({k:config[k] for k in ['dataset','backend','variant','parameter_count','train_samples','validation_samples','test_samples','device']})
    metrics.update(seed=seed,ablation=ablation,epochs=epochs,best_epoch=best_epoch,best_val_loss=best,train_seconds=train_seconds,validation_seconds=val_seconds,test_seconds=infer_seconds)
    shifted=logits-logits.max(1,keepdims=True);probs=np.exp(shifted);probs/=probs.sum(1,keepdims=True)
    with (out/'predictions.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f);writer.writerow(['test_id','true_label','predicted_label','confidence'])
        writer.writerows(zip(range(len(yt)),yt.tolist(),logits.argmax(1).tolist(),probs.max(1).tolist()))
    np.savez_compressed(out/'test_outputs.npz',logits=logits,labels=yt)
    np.savetxt(out/'confusion_matrix.csv',cm,delimiter=',',fmt='%d')
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    print('COMPLETE '+json.dumps(metrics),flush=True)
    return metrics

def train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    # A notebook and the CLI may request the same configuration concurrently.
    # Serialize that configuration; after waiting, reuse its complete results.
    from filelock import FileLock
    folder=experiment_dir(backend,dataset,variant,seed,ablation)
    folder.mkdir(parents=True,exist_ok=True)
    with FileLock(str(folder/'.train.lock'),timeout=7200):
        return _train(backend,dataset,variant,epochs,batch_size,seed,force,ablation)



In [5]:
def execute_dataset(name):
    rows=[]
    for variant in ['baseline','improved']:
        folder=ROOT/'results'/f'{name}_{BACKEND}_{variant}'
        if RETRAIN or not (folder/'metrics.json').exists():
            cmd=[sys.executable,'-m','src.train','--backend',BACKEND,'--dataset',name,'--variant',variant]
            if RETRAIN:cmd.append('--force')
            subprocess.run(cmd,cwd=ROOT,check=True)
        else:
            print('Đọc kết quả đã huấn luyện:',folder.name)
        rows.append(json.loads((folder/'metrics.json').read_text()))
        display(pd.read_csv(folder/'history.csv'))
    display(pd.DataFrame(rows)[['dataset','variant','accuracy','macro_f1','test_loss','top5_accuracy','best_epoch','train_seconds']])
    return rows

## 4. MNIST

In [6]:
mnist_results=execute_dataset('mnist')

Đọc kết quả đã huấn luyện: mnist_numpy_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.266781,0.922835,0.106439,0.965994,32.778638,1.463831
1,2,0.079065,0.975500,0.077075,0.976496,32.185802,1.702118
2,3,0.056552,0.983260,0.071810,0.978330,33.217386,1.497079
3,4,0.044776,0.986445,0.063222,0.980830,32.789730,1.827264
4,5,0.038266,0.987926,0.052263,0.983497,32.427002,1.506255


Đọc kết quả đã huấn luyện: mnist_numpy_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.258647,0.932853,0.079128,0.976663,132.629032,5.657167
1,2,0.081766,0.977297,0.056040,0.983331,131.304186,5.293225
2,3,0.054620,0.984389,0.053768,0.982830,133.261261,4.724603
3,4,0.044304,0.987037,0.048963,0.986331,152.143771,7.559457
4,5,0.037415,0.988722,0.045050,0.987331,205.383991,6.518152


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,mnist,baseline,0.9869,0.986795,0.040283,0.9998,5,163.398558
1,mnist,improved,0.9892,0.989140,0.031554,0.9999,5,754.722241


## 5. CIFAR-10

In [7]:
cifar10_results=execute_dataset('cifar10')

Đọc kết quả đã huấn luyện: cifar10_numpy_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.660106,0.404889,1.462729,0.4722,43.095112,1.901593
1,2,1.376718,0.512711,1.329451,0.5366,43.798373,2.297233
2,3,1.270960,0.552267,1.270386,0.5548,47.618070,2.172365
3,4,1.200416,0.578156,1.197988,0.5792,53.225548,2.160710
4,5,1.153476,0.594289,1.168176,0.5860,49.038745,2.391820
5,6,1.101936,0.613044,1.144686,0.6046,54.890898,2.053789
6,7,1.065234,0.626356,1.107762,0.6092,62.175512,5.128380
7,8,1.030150,0.639978,1.092641,0.6164,93.374897,3.903529
8,9,1.003459,0.650178,1.087300,0.6218,86.698737,3.805991
9,10,0.973473,0.657489,1.071399,0.6254,97.997694,4.685529


Đọc kết quả đã huấn luyện: cifar10_numpy_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.584361,0.434711,1.350125,0.5202,139.288201,3.355149
1,2,1.247750,0.557956,1.145852,0.5934,150.082531,6.436105
2,3,1.128135,0.600378,1.100304,0.6048,159.400132,6.971630
3,4,1.051660,0.626889,1.112664,0.5944,152.313783,5.165710
4,5,0.996992,0.647533,1.011546,0.6370,145.321995,6.506427
5,6,0.953492,0.662689,1.051300,0.6384,156.688070,6.085105
6,7,0.912124,0.678000,0.980786,0.6556,155.076159,5.697448
7,8,0.882590,0.689578,0.980368,0.6566,180.441290,8.639167
8,9,0.856651,0.694733,0.940905,0.6698,243.631772,6.949226
9,10,0.827364,0.707711,1.052874,0.6362,197.978958,6.521858


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar10,baseline,0.6315,0.628553,1.060018,0.9630,10,631.913585
1,cifar10,improved,0.6643,0.663441,0.965280,0.9683,9,1680.222891


## 6. CIFAR-100

In [8]:
cifar100_results=execute_dataset('cifar100')

Đọc kết quả đã huấn luyện: cifar100_numpy_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.176548,0.071689,3.768465,0.1264,108.148497,4.646153
1,2,3.574998,0.162978,3.441845,0.1764,129.803471,5.299595
2,3,3.323420,0.207044,3.312973,0.2064,113.311210,4.441986
3,4,3.177229,0.233378,3.208103,0.2222,106.523947,4.331324
4,5,3.066030,0.254022,3.139202,0.2310,104.365949,4.079418
5,6,2.992112,0.269156,3.122679,0.2422,88.337927,3.807207
6,7,2.915954,0.285844,3.068413,0.2552,87.908446,3.700579
7,8,2.860416,0.294022,3.083184,0.2412,88.548799,3.845535
8,9,2.810943,0.304444,2.989666,0.2586,86.960090,3.835714
9,10,2.768133,0.313667,2.966747,0.2692,99.916172,4.595437


Đọc kết quả đã huấn luyện: cifar100_numpy_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.133329,0.087511,3.650422,0.1564,232.953088,7.909337
1,2,3.549103,0.168178,3.293828,0.2188,205.820911,7.436608
2,3,3.272079,0.212244,3.094445,0.2508,183.683687,6.162401
3,4,3.102081,0.240111,2.997106,0.2600,160.925172,6.165379
4,5,2.966510,0.269356,2.906307,0.2778,155.920469,6.241059
5,6,2.872438,0.286822,2.837638,0.2898,180.315713,6.133990
6,7,2.794722,0.300622,2.803330,0.2974,166.821662,7.606551
7,8,2.740196,0.308022,2.775676,0.3022,182.067044,8.827997
8,9,2.668225,0.325978,2.744813,0.3078,167.258906,5.707535
9,10,2.628737,0.330933,2.818550,0.3000,135.319206,5.395623


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar100,baseline,0.2823,0.273536,2.922322,0.5795,12,1195.833606
1,cifar100,improved,0.3242,0.316807,2.680801,0.6302,12,2044.610536


## 7. Kiểm tra triển khai

In [9]:
verification=ROOT/'results'/f'verification_{BACKEND}.json'
if verification.exists():
    display(pd.DataFrame(json.loads(verification.read_text())))
else:
    print(subprocess.run([sys.executable,'tools/run_test_suite.py'],cwd=ROOT,capture_output=True,text=True,check=True).stderr)

test_dropout_identity (tests.test_ablation.AblationTests) ... ok
test_no_bn_removes_all_four (tests.test_ablation.AblationTests) ... ok
test_no_skip_preserves_branch_and_gradient (tests.test_ablation.AblationTests) ... ok
test_output_and_training_gradient (tests.test_ablation.AblationTests) ... ok
test_seed_and_ablation_paths_do_not_collide (tests.test_ablation.AblationTests) ... ok
test_shared_weights (tests.test_ablation.AblationTests) ... ok
test_bn_dense_gradients (tests.test_numpy.Gradients) ... ok
test_bn_gradients (tests.test_numpy.Gradients) ... ok
test_conv_gradients (tests.test_numpy.Gradients) ... ok
test_dense_gradients (tests.test_numpy.Gradients) ... ok
test_learning_tiny_problem (tests.test_numpy.Gradients) ... ok
test_loss_stability (tests.test_numpy.Gradients) ... ok
test_pool_gradients (tests.test_numpy.Gradients) ... ok
test_residual_gradient (tests.test_numpy.Gradients) ... ok
test_shapes_and_counts (tests.test_numpy.Gradients) ... ok

------------------------------

## Diễn giải

So sánh baseline với improved trong cùng dataset/backend. Accuracy không phản ánh toàn bộ chất lượng ở CIFAR-100: cần xem macro-F1, top-5 và các lớp hay nhầm. Train metrics được tích lũy trong lúc cập nhật trọng số, có dropout ở improved; validation chạy ở chế độ eval, nên không thể diễn giải mọi chênh lệch train-validation là overfitting.

Các chỉ số là một lượt chạy seed 42. Thời gian gồm bước huấn luyện thực tế nhưng không phải benchmark phần cứng độc lập. Notebook 05 đưa ra so sánh chung dựa trên 18 kết quả.